# Ridge Regression Model — Airbnb Price Predictor

In [1]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

## Load Data

In [3]:
# Read with Polars (fast parquet I/O), then convert to pandas for sklearn
df: pd.DataFrame = pl.read_parquet("/Users/snehasharma/Desktop/2450-final-project/airbnb-price-predictor/data/airbnb_cleaned.parquet").to_pandas()
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded dataset: 62,544 rows × 226 columns


**Findings:** The cleaned dataset contains 62,544 listings across NYC, LA, and Chicago with 226 columns after feature engineering. The parquet format enables fast I/O via Polars before converting to pandas for sklearn compatibility. This is our modeling-ready dataset — no additional cleaning steps are needed before fitting.

## Define Features

**Findings:** The feature set spans 217 numeric inputs — including 191 binary amenity dummies — and 5 categorical columns to be one-hot encoded. Neighbourhood is included as a categorical to capture the hyper-local price effects surfaced in EDA, where a single neighbourhood dummy can shift log-price by 0.5+. The high amenity dimensionality makes Ridge's L2 regularization especially appropriate: it shrinks correlated dummies toward zero without eliminating them entirely.

In [4]:
# Drop identifiers and the raw price columns (log_price is our target)
DROP_COLS = {"id", "host_id", "price_usd", "log_price"}

y = df["log_price"].copy()
X = df.drop(columns=list(DROP_COLS), errors="ignore")

# Categorical columns — will be one-hot encoded
CATEGORICAL_COLS = [
    "room_type",
    "property_type",
    "city",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",  # has ~12 % nulls → filled below
]

# Explicit numeric core features (already scaled in clean_data.py)
NUMERIC_CORE = [
    "latitude", "longitude", "accommodates", "bedrooms", "beds",
    "minimum_nights", "maximum_nights", "number_of_reviews",
    "review_scores_rating", "review_scores_accuracy",
    "review_scores_cleanliness", "review_scores_checkin",
    "review_scores_communication", "review_scores_location",
    "review_scores_value", "host_is_superhost", "host_listings_count",
    "host_identity_verified", "instant_bookable", "availability_365",
    "calculated_host_listings_count", "reviews_per_month",
    "bathrooms", "bathroom_shared", "years_as_host", "amenity_count",
]

# All binary amenity dummies
AMENITY_COLS = [c for c in X.columns if c.startswith("amenity_") and c != "amenity_count"]

NUMERIC_COLS = [c for c in NUMERIC_CORE + AMENITY_COLS if c in X.columns]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in X.columns]

# Keep only the columns we actually use (avoids stray cols sneaking into passthrough)
X = X[NUMERIC_COLS + CATEGORICAL_COLS].copy()

print(f"  Numeric features    : {len(NUMERIC_COLS)}")
print(f"  Categorical features: {len(CATEGORICAL_COLS)}")

  Numeric features    : 217
  Categorical features: 5


**Findings:** We hold out 20% (12,509 listings) as a test set, preserving 50,035 rows for training — large enough to support 5-fold CV with each fold containing ~10,000 samples. The split is done before any preprocessing to ensure the test set is fully unseen at every stage. `random_state=42` is fixed throughout to ensure reproducibility across experiments.

## Train / Test Split

**Findings:** The pipeline chains median imputation for numerics and constant-fill + one-hot encoding for categoricals into a single `ColumnTransformer`, then passes the dense matrix to `Ridge(alpha=1.0)`. Wrapping everything in a `Pipeline` ensures no leakage between preprocessing and model fitting during cross-validation. The `alpha=1.0` default is a starting point — next steps should include `RidgeCV` or a grid search to tune regularization strength.

In [5]:
# Split BEFORE any fitting to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")

Train size: 50,035  |  Test size: 12,509


**Findings:** 5-fold CV on the training set yields a mean R² of **0.7235 ± 0.0073**, indicating the model explains ~72% of log-price variance with low fold-to-fold variance. The tight spread (< 0.02) signals stable generalization — the model is not memorizing any particular data split. This is a strong linear baseline; the roadmap should include a tree-based ensemble (XGBoost or LightGBM) to capture the nonlinear structure that Ridge cannot model.

## Build Pipeline

**Findings:** On the held-out test set the model achieves **R² = 0.724**, **RMSE = 0.369 log-USD**, and **MAE = 0.283** — nearly identical to the CV estimate, confirming no overfitting. An RMSE of 0.37 in log space translates to predictions typically within a factor of e^0.37 ≈ 1.45× of the true price (roughly ±45%). This sets our baseline; the next model iteration should aim to push R² above 0.78 by adding interaction terms or switching to a gradient-boosted model.

In [6]:
# Numeric: median-impute the handful of columns with nulls (host_listings_count),
# then pass through — values are already scaled upstream
numeric_transformer = SimpleImputer(strategy="median")

# Categorical: fill nulls → one-hot encode
# handle_unknown='ignore' silently zeros out any unseen category at inference time
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_COLS),
        ("cat", categorical_transformer, CATEGORICAL_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,  # keeps names clean for coefficient extraction
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0)),
])

**Findings:** Neighbourhood dominates the top coefficients in both directions — Malibu and Pacific Palisades push prices up by ~0.6–0.7 log-units, while South Chicago, Watts, and Fuller Park pull them down by similar magnitudes. Shared room is the only room-type term in the top 15, with a large negative coefficient (−0.60), confirming EDA findings that room type is a primary price driver. This suggests location and room type together carry most of the signal, and that adding finer spatial features — walk score, transit access, distance to landmarks — could help differentiate listings *within* a neighbourhood.

## Cross-Validation

**Findings:** The fitted pipeline is serialized to `models/ridge_model.pkl` and can be loaded for inference or comparison without re-training. Future model versions should be saved with a versioned filename (e.g., `ridge_v2_alpha10.pkl`) alongside a metrics log so experiments remain reproducible and comparable over time.

In [7]:
# Evaluate before final fit to get an unbiased training-time estimate
cv_scores = cross_val_score(
    pipeline, X_train, y_train, cv=5, scoring="r2", n_jobs=-1
)
print(f"CV R² per fold : {[round(s, 4) for s in cv_scores]}")
print(f"CV R² mean ± std : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV R² per fold : [np.float64(0.7291), np.float64(0.7099), np.float64(0.7302), np.float64(0.7223), np.float64(0.726)]
CV R² mean ± std : 0.7235 ± 0.0073


## Train & Evaluate

In [8]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("── Test-set metrics ──")
print(f"  RMSE : {rmse:.4f}")
print(f"  MAE  : {mae:.4f}")
print(f"  R²   : {r2:.4f}")

── Test-set metrics ──
  RMSE : 0.3687
  MAE  : 0.2834
  R²   : 0.7240


## Top Coefficients

In [9]:
# Recover feature names from the ColumnTransformer after fitting
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients  = pipeline.named_steps["ridge"].coef_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)

print("── Top 15 Ridge coefficients (by |value|) ──")
coef_df.head(15)

── Top 15 Ridge coefficients (by |value|) ──


,feature,coefficient,abs_coef
0,neighbourhood_cleansed_Malibu,0.666646,0.666646
1,neighbourhood_cleansed_Pacific Palisades,0.605030,0.605030
2,room_type_Shared room,-0.598415,0.598415
3,neighbourhood_cleansed_South Chicago,-0.593603,0.593603
4,neighbourhood_cleansed_DUMBO,0.565706,0.565706
5,neighbourhood_cleansed_Beverly Crest,0.557156,0.557156
6,neighbourhood_cleansed_Hollywood Hills West,0.551435,0.551435
7,neighbourhood_cleansed_Avalon,0.549097,0.549097
8,neighbourhood_cleansed_Unincorporated Santa Mo...,0.519345,0.519345
9,neighbourhood_cleansed_Watts,-0.506810,0.506810


## Save Model

In [10]:
os.makedirs("models", exist_ok=True)
model_path = "models/ridge_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)
print(f"Pipeline saved → {model_path}")

Pipeline saved → models/ridge_model.pkl
